# SO101 MDH Forward Kinematics Demo

This notebook demonstrates:
1. **MDH Parameters** — Modified DH (Craig convention) parameter table for SO101 (6-DOF)
2. **URDF Visualization** — Visualize the default configuration $q_{init} = [0, 0, 0, 0, 0, 0]$
3. **Animation** — Animate joint-by-joint motion from default config, verify URDF and MDH match

> Step-by-step MDH matrix derivation for each joint: see [so101_mdh_derivation.md](so101_mdh_derivation.md)

> **Note:** Unlike Panda (whose URDF frames follow MDH rules, parameters can be read directly), SO101's URDF is CAD-exported (onshape-to-robot), frames do not follow DH conventions. The MDH parameters below were obtained via **geometric analysis + numerical optimization**. See `so101_urdf_to_dh.py` for the extraction process.

In [ ]:
import numpy as np
from numpy import pi, cos, sin
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display

# Chinese font setup
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=6, suppress=True)

%matplotlib inline

---
## 1. SO101 MDH Parameter Table

### Modified DH (Craig Convention)

The transformation from frame $i{-}1$ to frame $i$ is:

$${}^{i-1}T_i = \text{Rot}_x(\alpha_{i-1}) \cdot \text{Trans}_x(a_{i-1}) \cdot \text{Rot}_z(\theta_i) \cdot \text{Trans}_z(d_i)$$

### Parameter Table

| Joint | $a_{i-1}$ (m) | $\alpha_{i-1}$ (rad) | $d_i$ (m) | $\theta_i$ (rad) | Joint Limits (rad) |
|:-----:|:--------------:|:--------------------:|:---------:|:-----------------:|:------------------:|
| 1 (shoulder_pan) | 0.0388 | $+\pi$ | $-0.1166$ | $\theta_1 + 0$ (init: 0) | $[-1.92,\; +1.92]$ |
| 2 (shoulder_lift) | 0.0304 | $+\pi/2$ | $\approx 0$ | $\theta_2 - 1.327$ (init: 0) | $[-1.75,\; +1.75]$ |
| 3 (elbow_flex) | 0.1160 | $0$ | $\approx 0$ | $\theta_3 + 1.289$ (init: 0) | $[-1.69,\; +1.69]$ |
| 4 (wrist_flex) | 0.1350 | $0$ | $\approx 0$ | $\theta_4 + 1.609$ (init: 0) | $[-1.66,\; +1.66]$ |
| 5 (wrist_roll) | 0 | $-\pi/2$ | $-0.0845$ | $\theta_5 - 3.190$ (init: 0) | $[-2.74,\; +2.84]$ |
| 6 (gripper) | 0.0202 | $+\pi/2$ | $-0.0188$ | $\theta_6 + 0$ (init: 0) | $[-0.17,\; +1.75]$ |

> **Theta offsets:** SO101 has $\theta_{\text{off}}$ because the URDF zero configuration does not correspond to the DH zero configuration. The actual DH angle is $\theta_i^{\text{DH}} = \theta_i^{\text{URDF}} + \theta_{\text{off},i}$.

> **Parameter source:** Geometric analysis + numerical optimization from URDF. See `so101_urdf_to_dh.py`.

In [ ]:
# ============================================================
#  URDF FK (ground truth)
# ============================================================
def rpy_to_rotation(roll, pitch, yaw):
    cr, sr = cos(roll), sin(roll)
    cp, sp = cos(pitch), sin(pitch)
    cy, sy = cos(yaw), sin(yaw)
    return np.array([
        [cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr],
        [sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr],
        [-sp,   cp*sr,            cp*cr            ]
    ])

def urdf_joint_transform(xyz, rpy, theta, axis=(0, 0, 1)):
    T = np.eye(4)
    T[:3, 3] = xyz
    T[:3, :3] = rpy_to_rotation(*rpy)
    ax, ay, az = axis
    ct, st = cos(theta), sin(theta)
    K = np.array([[0, -az, ay], [az, 0, -ax], [-ay, ax, 0]])
    R = np.eye(3) + st * K + (1 - ct) * (K @ K)
    T_j = np.eye(4)
    T_j[:3, :3] = R
    return T @ T_j

# SO101 URDF joint definitions (6 revolute joints)
so101_joints = [
    ('shoulder_pan',  (0.0388353, -8.97657e-09, 0.0624), (3.14159, 4.18253e-17, -3.14159),  (0,0,1)),
    ('shoulder_lift', (-0.0303992, -0.0182778, -0.0542), (-1.5708, -1.5708, 0),              (0,0,1)),
    ('elbow_flex',    (-0.11257, -0.028, 1.73763e-16),   (-3.636e-16, 8.743e-16, 1.5708),    (0,0,1)),
    ('wrist_flex',    (-0.1349, 0.0052, 3.62355e-17),    (4.025e-15, 8.674e-16, -1.5708),    (0,0,1)),
    ('wrist_roll',    (5.55e-17, -0.0611, 0.0181),       (1.5708, 0.0486795, 3.14159),       (0,0,1)),
    ('gripper',       (0.0202, 0.0188, -0.0234),         (1.5708, -5.24284e-08, -1.41553e-15),(0,0,1)),
]
N = 6
joint_names_list = [j[0] for j in so101_joints]

def fk_urdf(q):
    """URDF FK. Returns (T_end, frames_list)."""
    T = np.eye(4)
    frames = [np.eye(4)]
    for i, (_, xyz, rpy, axis) in enumerate(so101_joints):
        T = T @ urdf_joint_transform(xyz, rpy, q[i], axis)
        frames.append(T.copy())
    return T, frames

# ============================================================
#  MDH Parameters (from optimization) & Forward Kinematics
# ============================================================
# (a_{i-1}, alpha_{i-1}, d_i, theta_offset_i)
mdh_params = [
    (0.038835,  pi,     -0.116600,  0.0000),   # Joint 1: shoulder_pan  (init: 0)
    (0.030399,  pi/2,   -0.000223, -1.3270),   # Joint 2: shoulder_lift (init: 0)
    (0.116000,  0.0,     0.000067,  1.2885),   # Joint 3: elbow_flex   (init: 0)
    (0.135000,  0.0,    -0.000022,  1.6093),   # Joint 4: wrist_flex   (init: 0)
    (0.000000, -pi/2,   -0.084500, -3.1903),   # Joint 5: wrist_roll   (init: 0)
    (0.020200,  pi/2,   -0.018800,  0.0000),   # Joint 6: gripper      (init: 0)
]

# Default initial joint values (all zeros for SO101)
q_init = np.zeros(N)


def mdh_transform(theta, d, a_prev, alpha_prev):
    """
    MDH single-joint transform:
    T_i = Rx(alpha_{i-1}) * Tx(a_{i-1}) * Rz(theta_i) * Tz(d_i)
    """
    ca, sa = cos(alpha_prev), sin(alpha_prev)
    ct, st = cos(theta), sin(theta)

    Rx = np.array([[1, 0,  0,  0],
                   [0, ca, -sa, 0],
                   [0, sa,  ca, 0],
                   [0, 0,   0,  1]])
    Tx = np.array([[1, 0, 0, a_prev],
                   [0, 1, 0, 0],
                   [0, 0, 1, 0],
                   [0, 0, 0, 1]])
    Rz = np.array([[ct, -st, 0, 0],
                   [st,  ct, 0, 0],
                   [0,   0,  1, 0],
                   [0,   0,  0, 1]])
    Tz = np.array([[1, 0, 0, 0],
                   [0, 1, 0, 0],
                   [0, 0, 1, d],
                   [0, 0, 0, 1]])
    return Rx @ Tx @ Rz @ Tz


def fk_mdh(q):
    """MDH FK with theta offsets. Returns (T_end, frames_list)."""
    frames = [np.eye(4)]
    T = np.eye(4)
    for i, (a, alpha, d, theta_off) in enumerate(mdh_params):
        T = T @ mdh_transform(q[i] + theta_off, d, a, alpha)
        frames.append(T.copy())
    return T, frames


print('MDH parameters defined: 6 revolute joints')
print(f'Total DOF: {N}')
print(f'Default init: q = {q_init}')
print(f'\nParameter table:')
print(f'{"Joint":>18} | {"a (m)":>10} | {"alpha (rad)":>12} | {"d (m)":>10} | theta_off')
print('-' * 80)
for i, (a, alpha, d, theta_off) in enumerate(mdh_params):
    alpha_str = f'{alpha/pi:+.1f}pi' if abs(alpha) > 1e-6 else '0'
    off_str = f'{theta_off:+.4f}' if abs(theta_off) > 1e-6 else '0'
    print(f'{joint_names_list[i]:>18} | {a:>10.4f} | {alpha_str:>12} | {d:>10.4f} | {off_str}')

---
## 2. Visualize Default Configuration

Visualize the default configuration $\mathbf{q}_{\text{init}} = [0, 0, 0, 0, 0, 0]$ and compare MDH FK vs URDF FK.

> SO101 all joint limits include 0, so the default configuration is all zeros.

In [ ]:
# ============================================================
#  Visualize default configuration: URDF vs MDH
# ============================================================

# MDH frames
T_mdh_end, frames_mdh = fk_mdh(q_init)
pos_mdh = np.array([T[:3, 3] for T in frames_mdh])

# URDF frames
T_urdf_end, frames_urdf = fk_urdf(q_init)
pos_urdf_end = T_urdf_end[:3, 3]

# Verify match
pos_err = np.linalg.norm(pos_mdh[-1] - pos_urdf_end)
print(f'Default config q_init = {q_init}')
print(f'End-effector comparison:')
print(f'  MDH  end: {pos_mdh[-1]}')
print(f'  URDF end: {pos_urdf_end}')
print(f'  Error:    {pos_err:.2e} m  {"PASS" if pos_err < 1e-6 else "FAIL"}')

# --- Plot ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# MDH arm
ax.plot(pos_mdh[:, 0], pos_mdh[:, 1], pos_mdh[:, 2],
        '-o', color='#2196F3', linewidth=3, markersize=8,
        label='MDH FK', markerfacecolor='#2196F3',
        markeredgecolor='white', markeredgewidth=1)

# URDF end-effector marker
ax.scatter(*pos_urdf_end, color='#FF5722', s=150, marker='*',
           zorder=10, label='URDF end-effector')

# Base marker
ax.scatter(*pos_mdh[0], color='#4CAF50', s=120, marker='s',
           zorder=5, label='Base')

# Joint labels
joint_labels = ['Base', 'J1', 'J2', 'J3', 'J4', 'J5', 'J6']
for i, (name, pos) in enumerate(zip(joint_labels, pos_mdh)):
    ax.text(pos[0]+0.01, pos[1]+0.01, pos[2]+0.01, name, fontsize=8, color='#333')

q_str = ', '.join(f'{v}' for v in q_init)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title(f'SO101 Default Configuration  $q_{{init}}$ = [{q_str}]', fontsize=13)
ax.set_xlim([-0.3, 0.3])
ax.set_ylim([-0.3, 0.3])
ax.set_zlim([-0.15, 0.25])
ax.view_init(elev=25, azim=135)
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

---
## 3. Animation: Joint-by-Joint Motion + MDH Verification

Starting from the default configuration $\mathbf{q}_{\text{init}} = [0, 0, 0, 0, 0, 0]$, each joint moves independently (sinusoidal trajectory). The animation shows:
- **Blue line**: MDH forward kinematics
- **Red star**: URDF end-effector position (verification)
- **Top-right info**: MDH vs URDF position error at the end-effector

In [ ]:
# ============================================================
#  Generate trajectory: each joint moves one at a time
#  Starting from default config q_init
# ============================================================
n_steps = 20       # frames per joint
n_joints = N
targets = [pi/4, -pi/3, pi/4, -pi/2, pi/3, pi/4]  # peak displacement per joint

trajectory = [q_init.copy()]  # start at default config
for j in range(n_joints):
    for k in range(1, n_steps + 1):
        t = k / n_steps
        q = q_init.copy()
        q[j] = q_init[j] + targets[j] * sin(pi * t)  # oscillate around init value
        trajectory.append(q.copy())

# Pre-compute all frames
all_mdh_pos = []
all_urdf_end = []
all_errors = []

for q in trajectory:
    T_mdh, frames = fk_mdh(q)
    pos = np.array([T[:3, 3] for T in frames])
    all_mdh_pos.append(pos)
    
    T_urdf, _ = fk_urdf(q)
    urdf_end = T_urdf[:3, 3]
    all_urdf_end.append(urdf_end)
    
    err = np.linalg.norm(pos[-1] - urdf_end)
    all_errors.append(err)

max_err = max(all_errors)
print(f'Trajectory: {len(trajectory)} frames ({n_steps} per joint x {n_joints} joints + 1)')
print(f'Starting from q_init = {q_init}')
print(f'Max MDH vs URDF end-effector error: {max_err:.2e} m  {"PASS" if max_err < 1e-6 else "FAIL"}')

In [ ]:
# ============================================================
#  Create animation (single 3D view, error as text overlay)
# ============================================================
fig = plt.figure(figsize=(10, 8))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.set_xlabel('X (m)')
ax3d.set_ylabel('Y (m)')
ax3d.set_zlabel('Z (m)')
ax3d.set_title('SO101 Joint-by-Joint Motion (from $q_{init}$)', fontsize=13, fontweight='bold')
ax3d.set_xlim([-0.35, 0.35])
ax3d.set_ylim([-0.35, 0.35])
ax3d.set_zlim([-0.2, 0.3])
ax3d.view_init(elev=25, azim=135)

# Plot elements
line_mdh, = ax3d.plot([], [], [], '-o', color='#2196F3', linewidth=3,
                       markersize=6, label='MDH FK',
                       markerfacecolor='#2196F3', markeredgecolor='white', markeredgewidth=0.5)
scat_urdf = ax3d.scatter([], [], [], color='#FF5722', s=120, marker='*',
                          zorder=10, label='URDF end')
ax3d.legend(fontsize=10, loc='upper left')

# Info text (top-left)
info_text = ax3d.text2D(0.02, 0.95, '', transform=ax3d.transAxes, fontsize=9,
                         verticalalignment='top', fontfamily='monospace',
                         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Error text (top-right)
err_text = ax3d.text2D(0.98, 0.95, '', transform=ax3d.transAxes, fontsize=9,
                        verticalalignment='top', horizontalalignment='right',
                        fontfamily='monospace',
                        bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.8))

def init():
    line_mdh.set_data_3d([], [], [])
    info_text.set_text('')
    err_text.set_text('')
    return [line_mdh, info_text, err_text]

def update(idx):
    p = all_mdh_pos[idx]
    ue = all_urdf_end[idx]
    q = trajectory[idx]
    err = all_errors[idx]
    active = min((idx - 1) // n_steps, n_joints - 1) if idx > 0 else 0

    line_mdh.set_data_3d(p[:, 0], p[:, 1], p[:, 2])
    scat_urdf._offsets3d = ([ue[0]], [ue[1]], [ue[2]])

    q_str = ', '.join(f'{v:+.2f}' for v in q)
    info_text.set_text(
        f'Frame: {idx}/{len(trajectory)-1}\n'
        f'Active: {joint_names_list[active]}\n'
        f'q = [{q_str}]'
    )

    err_text.set_text(
        f'MDH vs URDF\n'
        f'End-eff err: {err:.2e} m\n'
        f'Max err: {max_err:.2e} m'
    )

    return [line_mdh, info_text, err_text]

anim = FuncAnimation(fig, update, frames=len(trajectory),
                     init_func=init, interval=80, blit=False)
plt.tight_layout()
plt.close(fig)

# Display as interactive JS animation
display(HTML(anim.to_jshtml()))